In [0]:
pip install xgboost

In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, StructType, StructField, ArrayType, DoubleType, IntegerType
import pandas as pd
from pyarrow import *
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression  
from pyspark.ml.feature import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import xgboost as xgb
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pyarrow.parquet as pq

In [0]:
xgb_df = spark.read.table("hive_metastore.default.lreg_df")

#### XGB

In [0]:
final_df = xgb_df.withColumn("AMPM_flag", when(col("AM_PM")=="PM", 1.0).otherwise(0.0))

In [0]:
final_df = final_df.drop("AO","NOME","TIPOINST","TAGCOM","REDE","ID_prefix","ID_OBJECTO")

In [0]:
from pyspark.sql import functions as F, Window
ID_COL, DATE_COL = "ID", "DATE"

w = Window.partitionBy(ID_COL).orderBy(F.col(DATE_COL))
for c in ["INTENSITY","TENSION"]:
    # extra short lags
    for k in [2, 3, 6, 12]:  # e.g., t-30m, -45m, -1.5h, -3h (if 15-min cadence)
        final_df = final_df.withColumn(f"{c}_lag{k}", F.lag(F.col(c), k).over(w))
    # short rolling means
    for k in [4, 8, 24]:
        final_df = final_df.withColumn(f"{c}_rmean{k}", F.avg(F.col(c)).over(w.rowsBetween(-k+1, 0)))


In [0]:
from pyspark.sql import functions as F

ID_COL = "ID"
min_count = 10  # tune: bucket IDs with <10 rows
id_freq = final_df.groupBy(ID_COL).count()

final_df = (final_df
            .join(id_freq, ID_COL, "left")
            .withColumn(ID_COL, F.when(F.col("count") < min_count, F.lit("OTHER")).otherwise(F.col(ID_COL)))
            .drop("count"))


In [0]:
import builtins
import datetime as dt
from pyspark.sql import functions as F

DATE_COL, LABEL_COL = "DATE", "has_falha"

# already have train_df/test_df from your day split
trmm = (train_df.agg(F.to_date(F.min(DATE_COL)).alias("min_day"),
                     F.to_date(F.max(DATE_COL)).alias("max_day")).first())
tr_min, tr_max = trmm["min_day"], trmm["max_day"]
tr_days = (tr_max - tr_min).days + 1

# >>> use builtins.max here <<<
val_days = builtins.max(1, int(tr_days * 0.1))
val_start = tr_max - dt.timedelta(days=val_days - 1)

train_tr = train_df.filter(F.to_date(F.col(DATE_COL)) <  F.lit(val_start.isoformat()))
train_va = train_df.filter(F.to_date(F.col(DATE_COL)) >= F.lit(val_start.isoformat()))
train_tr_flag = train_df.withColumn(
    "is_val", (F.to_date(F.col(DATE_COL)) >= F.lit(val_start.isoformat())).cast("int")
)

# class weight from train_tr only
pos_neg = train_tr.agg(
    F.sum(F.col(LABEL_COL).cast("int")).alias("pos"),
    (F.count("*") - F.sum(F.col(LABEL_COL).cast("int"))).alias("neg")
).first()
scale_pos_weight = (pos_neg["neg"] / builtins.max(1, pos_neg["pos"])) if pos_neg["pos"] else 1.0

print("scale_pos_weight =", scale_pos_weight)
print(f"Validation window: {val_start} → {tr_max}")


In [0]:
# Build both variants with your make_xgb_pipeline(...) that uses OHE
pipe_w,  _ = make_xgb_pipeline(
    use_weather=True,  label_col=LABEL_COL,
    learningRate=0.06,     # a bit lower
    maxDepth=6,            # shallower trees reduce overfit on sparse OHE
    subsample=0.8,
    colsampleBytree=0.6,   # stronger column subsampling helps on wide sparse
    numRound=1200,         # allow more rounds with early stopping
    regAlpha=0.0,
    regLambda=2.0,         # stronger L2
    evalMetric="aucpr",
    treeMethod="hist",
    scalePosWeight=scale_pos_weight
)
pipe_nw, _ = make_xgb_pipeline(
    use_weather=False, label_col=LABEL_COL,
    learningRate=0.06, maxDepth=6, subsample=0.8, colsampleBytree=0.6,
    numRound=1200, regAlpha=0.0, regLambda=2.0, evalMetric="aucpr",
    treeMethod="hist", scalePosWeight=scale_pos_weight
)

# Fit with early stopping using validation indicator
model_w = pipe_w.fit(train_tr_flag, params={
    "validation_indicator_col": "is_val",
    "early_stopping_rounds": 100,
    # additional overfit guards:
    "min_child_weight": 5,
    "gamma": 1.0
})
model_nw = pipe_nw.fit(train_tr_flag, params={
    "validation_indicator_col": "is_val",
    "early_stopping_rounds": 100,
    "min_child_weight": 5,
    "gamma": 1.0
})


In [0]:
# Fixed column sets
WEATHER_COLS = ["temperature", "precipitation", "wind_speed", "humidity"]

BASE_NUMERIC = [
    "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
    "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
    "EVENT_COUNT_I","EVENT_COUNT_T",
    "TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK","DAY_OF_MONTH","DAY_OF_YEAR","HOUR_OF_DAY",
    "AMPM_flag"  # 0/1; we scale it together with the rest to keep the vector layout identical
]

def make_pipeline(use_weather: bool,
                  regParam=0.0, elasticNetParam=0.0, maxIter=100,
                  id_col="ID", conc_col="CONCELHO", label_col="has_falha"):
    # 1) categorical → index → OHE (dropLast=True for a canonical basis)
    idx_id   = StringIndexer(inputCol=id_col,   outputCol="ID_idx",        handleInvalid="keep")
    idx_conc = StringIndexer(inputCol=conc_col, outputCol="CONCELHO_idx",  handleInvalid="keep")
    ohe = OneHotEncoder(
        inputCols=["ID_idx","CONCELHO_idx"],
        outputCols=["ID_ohe","CONCELHO_ohe"],
        dropLast=True
    )

    # 2) numeric → assembler → scaler (stable names)
    numeric_cols = BASE_NUMERIC + (WEATHER_COLS if use_weather else [])
    num_asm  = VectorAssembler(inputCols=numeric_cols, outputCol="num_features")
    scaler   = StandardScaler(inputCol="num_features", outputCol="num_scaled",
                              withMean=True, withStd=True)

    # 3) final features (freeze order!)
    feats = VectorAssembler(inputCols=["ID_ohe","CONCELHO_ohe","num_scaled"], outputCol="features")

    # 4) LR
    lr = LogisticRegression(
        featuresCol="features", labelCol=label_col,
        regParam=regParam, elasticNetParam=elasticNetParam,
        maxIter=maxIter, standardization=False
    )
    # (standardization=False because we already scaled numerics; OHE parts don’t need scaling)

    return Pipeline(stages=[idx_id, idx_conc, ohe, num_asm, scaler, feats, lr]), numeric_cols


In [0]:
# --- Imports (Databricks / PySpark + XGBoost Spark) ---
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from xgboost.spark import SparkXGBClassifier
from pyspark.mllib.evaluation import BinaryClassificationMetrics
import builtins

# ==== Fixed column sets (same as LR) ====
WEATHER_COLS = ["temperature", "precipitation", "wind_speed", "humidity"]

BASE_NUMERIC = [
    "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
    "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
    "EVENT_COUNT_I","EVENT_COUNT_T",
    "TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK","DAY_OF_MONTH","DAY_OF_YEAR","HOUR_OF_DAY",
    "AMPM_flag"  # keep for stable layout
]

# ==== Pipeline builder (mirrors LR, swaps final stage to XGB) ====
def make_xgb_pipeline(use_weather: bool,
                      id_col="ID", conc_col="CONCELHO", label_col="has_falha",
                      # reasonable starting params, we’ll tune later
                      learningRate=0.1, maxDepth=8, subsample=0.8, colsampleBytree=0.8, numRound=300,
                      regAlpha=0.0, regLambda=1.0, evalMetric="aucpr", treeMethod="hist",
                      scalePosWeight=None):
    # 1) Categorical → Index → OHE (dropLast=True to match your LR research)
    idx_id   = StringIndexer(inputCol=id_col,   outputCol="ID_idx",        handleInvalid="keep")
    idx_conc = StringIndexer(inputCol=conc_col, outputCol="CONCELHO_idx",  handleInvalid="keep")
    ohe = OneHotEncoder(
        inputCols=["ID_idx","CONCELHO_idx"],
        outputCols=["ID_ohe","CONCELHO_ohe"],
        dropLast=True
    )

    # 2) Numeric → assembler → scaler (we keep scaling to preserve the exact vector layout)
    numeric_cols = BASE_NUMERIC + (WEATHER_COLS if use_weather else [])
    num_asm  = VectorAssembler(inputCols=numeric_cols, outputCol="num_features")
    scaler   = StandardScaler(inputCol="num_features", outputCol="num_scaled",
                              withMean=True, withStd=True)

    # 3) Final features (freeze order identical to LR)
    feats = VectorAssembler(inputCols=["ID_ohe","CONCELHO_ohe","num_scaled"], outputCol="features")

    # 4) XGBoost classifier (Spark API) — snake_case param names
    xgb = SparkXGBClassifier(
        features_col="features",
        label_col=label_col,
        prediction_col="prediction",
        probability_col="probability",
        raw_prediction_col="rawPrediction",

        learning_rate=learningRate,
        max_depth=maxDepth,
        subsample=subsample,
        colsample_bytree=colsampleBytree,
        num_round=numRound,
        reg_alpha=regAlpha,
        reg_lambda=regLambda,
        eval_metric=evalMetric,
        tree_method=treeMethod,

        num_workers=builtins.max(1, spark.sparkContext.defaultParallelism // 2)
    )
    if scalePosWeight is not None:
        xgb = xgb.setParams(scale_pos_weight=scalePosWeight)

    return Pipeline(stages=[idx_id, idx_conc, ohe, num_asm, scaler, feats, xgb]), numeric_cols


In [0]:
from pyspark.sql import functions as F
import datetime as dt

DATE_COL = "DATE"
LABEL_COL = "has_falha"

# 1) min/max days
mm = (final_df
      .agg(F.to_date(F.min(DATE_COL)).alias("min_day"),
           F.to_date(F.max(DATE_COL)).alias("max_day"))
      .collect()[0])

min_day = mm["min_day"]  # python date
max_day = mm["max_day"]

# 2) day-count and 80% cutoff day
total_days = (max_day - min_day).days + 1
train_days = int(total_days * 0.8)               # floor
cutoff_day = min_day + dt.timedelta(days=train_days - 1)  # inclusive

print(f"Range: {min_day} → {max_day}  | days={total_days}  | 80% cutoff (inclusive) = {cutoff_day}")

# 3) split on whole days (train: ≤ cutoff_day, test: > cutoff_day)
train_df = final_df.filter(F.to_date(F.col(DATE_COL)) <= F.lit(cutoff_day.isoformat())).cache()
test_df  = final_df.filter(F.to_date(F.col(DATE_COL))  > F.lit(cutoff_day.isoformat())).cache()

display(train_df.limit(5))
print("Train rows:", train_df.count(), " | Test rows:", test_df.count())

In [0]:
# ==== Build both variants ====
pipe_w,  num_cols_w  = make_xgb_pipeline(use_weather=True,  label_col=LABEL_COL)
pipe_nw, num_cols_nw = make_xgb_pipeline(use_weather=False, label_col=LABEL_COL)

# ==== Fit ====
model_w  = pipe_w.fit(train_df)
model_nw = pipe_nw.fit(train_df)

# ==== Predict ====
pred_w  = model_w.transform(test_df).cache()
pred_nw = model_nw.transform(test_df).cache()
display(pred_w.select("probability","prediction", LABEL_COL).limit(10))
display(pred_nw.select("probability","prediction", LABEL_COL).limit(10))



In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import functions as F

def eval_binary(pred_df, label_col=LABEL_COL, prob_col="probability", thr=0.5):
    # Convert ML vector -> array so we can index [1]
    scored = (pred_df
              .withColumn("p_arr", vector_to_array(F.col(prob_col)))
              .withColumn("p1", F.col("p_arr")[1])
              .withColumn("pred_thr", (F.col("p1") >= F.lit(thr)).cast("int")))

    # AUPRC / AUROC
    rdd = scored.select(F.col("p1").cast("double"), F.col(label_col).cast("double")).rdd.map(tuple)
    m = BinaryClassificationMetrics(rdd)
    auprc, auroc = m.areaUnderPR, m.areaUnderROC

    # Confusion matrix pieces
    tp = scored.filter((F.col(label_col)==1) & (F.col("pred_thr")==1)).count()
    fp = scored.filter((F.col(label_col)==0) & (F.col("pred_thr")==1)).count()
    fn = scored.filter((F.col(label_col)==1) & (F.col("pred_thr")==0)).count()
    tn = scored.filter((F.col(label_col)==0) & (F.col("pred_thr")==0)).count()

    precision = tp/(tp+fp) if (tp+fp) else 0.0
    recall    = tp/(tp+fn) if (tp+fn) else 0.0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0

    display(spark.createDataFrame([(thr, precision, recall, f1, tp, fp, fn, tn, auprc, auroc)],
                                  ["threshold","precision","recall","f1","TP","FP","FN","TN","AUPRC","AUROC"]))
    return {"AUPRC": auprc, "AUROC": auroc, "precision": precision, "recall": recall, "f1": f1}

def sweep_thresholds(pred_df, thresholds=(0.85,0.80,0.75,0.70,0.65,0.60), label_col=LABEL_COL):
    rows = []
    for t in thresholds:
        s = eval_binary(pred_df, label_col=label_col, thr=t)
        rows.append((t, s["precision"], s["recall"], s["f1"], s["AUPRC"], s["AUROC"]))
    out = spark.createDataFrame(rows, ["threshold","precision","recall","f1","AUPRC","AUROC"])
    display(out.orderBy(F.desc("threshold")))
    return out


In [0]:
print("=== WITH WEATHER @0.5 ===")
m_w = eval_binary(pred_w, thr=0.5)

print("=== NO WEATHER @0.5 ===")
m_nw = eval_binary(pred_nw, thr=0.5)

print("=== Threshold sweep — WITH WEATHER ===")
tbl_w = sweep_thresholds(pred_w)

print("=== Threshold sweep — NO WEATHER ===")
tbl_nw = sweep_thresholds(pred_nw)


In [0]:
# ==== Feature importance (aligned to vector metadata) ====
def _vector_attr_names(df_with_vec, vec_col):
    meta = df_with_vec.schema[vec_col].metadata
    names = []
    if "ml_attr" in meta and "attrs" in meta["ml_attr"]:
        for k in ["binary","nominal","numeric"]:
            for a in meta["ml_attr"]["attrs"].get(k, []):
                names.append((a["idx"], a["name"]))
    names = [n for _, n in sorted(names, key=lambda x: x[0])]
    return names

def extract_xgb_importance(pipeline_model, vec_col="features"):
    xgbm = pipeline_model.stages[-1]
    # getScore returns dict like {"f0": gain, "f1": gain, ...}
    score = xgbm.getBooster().getScore(importance_type="gain")
    tmp = pipeline_model.transform(train_df.limit(1)).select(vec_col)
    feat_names = _vector_attr_names(tmp, vec_col)
    rows = []
    for i, name in enumerate(feat_names):
        rows.append((i, name, float(score.get(f"f{i}", 0.0))))
    return spark.createDataFrame(rows, ["idx","feature","gain"]).orderBy(F.desc("gain"))

imp_w  = extract_xgb_importance(model_w)
imp_nw = extract_xgb_importance(model_nw)

print("Top 25 — with weather")
display(imp_w.limit(25))
print("Top 25 — no weather")
display(imp_nw.limit(25))



In [0]:
# ==== Grouped importance (ID / CONCELHO / num_weather / num_non_weather) ====
num_weather_set    = set(WEATHER_COLS)
def group_for_feature(fname: str):
    if fname.startswith("ID_ohe"):
        return "ID"
    if fname.startswith("CONCELHO_ohe"):
        return "CONCELHO"
    # numeric entries keep original col name in our layout (inside num_scaled)
    base = fname  # assembler preserves numeric col names
    return "num_weather" if base in num_weather_set else "num_non_weather"

def grouped_imp(imp_df):
    g = imp_df.withColumn("group",
                          F.when(F.col("feature").startswith("ID_ohe"), "ID")
                           .when(F.col("feature").startswith("CONCELHO_ohe"), "CONCELHO")
                           .when(F.col("feature").isin(list(num_weather_set)), "num_weather")
                           .otherwise("num_non_weather"))
    out = g.groupBy("group").agg(
        F.count("*").alias("n_dims"),
        F.sum("gain").alias("sum_gain"),
        F.avg("gain").alias("mean_gain")
    ).orderBy(F.desc("sum_gain"))
    return out

print("Grouped — with weather")
display(grouped_imp(imp_w))

print("Grouped — no weather")
display(grouped_imp(imp_nw))

In [0]:
train = df_transformed.filter(col("DATE") < "2023-11-30")
test  = df_transformed.filter(col("DATE") >= "2023-11-30")

In [0]:
feature_cols = [
    "INTENSITY", "TENSION", "H_LIM_I", "H_LIM_T",
    "MAVERAGE_2H_I", "MAVERAGE_2H_T", "MAVERAGE_1D_I", "MAVERAGE_1D_T",
    "EVENT_COUNT_I", "EVENT_COUNT_T", "TIME_OVER_LIMIT_I", "TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK", "DAY_OF_MONTH", "DAY_OF_YEAR", "HOUR_OF_DAY",
    "ID_idx", "AM_PM_idx", "CONCELHO_idx"
]

train_pd = train.select(feature_cols + ["has_falha"]).toPandas()
test_pd  = test.select(feature_cols + ["has_falha"]).toPandas()

In [0]:
X_train = train_pd[feature_cols]
y_train = train_pd["has_falha"]

X_test = test_pd[feature_cols]
y_test = test_pd["has_falha"]


In [0]:


model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    use_label_encoder=False
)
model.fit(X_train, y_train)

_ = evaluate_estimator_binary(model, X_test, y_test,
                              model_name="xgb_baseline",
                              base_dir="/dbfs/reports")


In [0]:
model.fit(X_train, y_train)

#### Analysis

In [0]:
y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

In [0]:
import matplotlib.pyplot as plt
xgb.plot_importance(model, max_num_features=20)
plt.show()

In [0]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix
)

def _metrics_at_threshold(y_true, y_prob, thr: float):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total = tp + tn + fp + fn
    acc   = (tp + tn) / total if total else float('nan')
    prec  = tp / (tp + fp) if (tp + fp) else 0.0
    rec   = tp / (tp + fn) if (tp + fn) else 0.0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    spec  = tn / (tn + fp) if (tn + fp) else 0.0
    bal   = (rec + spec) / 2
    denom = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc   = ((tp*tn - fp*fn) / denom) if denom else 0.0
    return dict(thr=float(thr), tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
                acc=acc, prec=prec, rec=rec, f1=f1, spec=spec, bal_acc=bal, mcc=mcc)

def _save_cm(cm_dict, title, out_png):
    cm = np.array([[cm_dict['tn'], cm_dict['fp']],
                   [cm_dict['fn'], cm_dict['tp']]], dtype=int)
    fig, ax = plt.subplots(figsize=(4, 4), dpi=160)
    im = ax.imshow(cm, interpolation='nearest')
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['0', '1'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['0', '1'])
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches='tight'); plt.close(fig)

def evaluate_estimator_binary(estimator, X_test, y_test,
                              model_name="model_v1",
                              base_dir="/dbfs/reports"):
    """
    Works with any sklearn-like estimator that has predict_proba(X)[:,1].
    Saves artifacts and prints a compact summary.
    """
    os.makedirs(base_dir, exist_ok=True)
    outdir = os.path.join(base_dir, model_name)
    os.makedirs(outdir, exist_ok=True)

    # --- Probabilities
    y_prob = estimator.predict_proba(X_test)[:, 1]
    y_true = np.asarray(y_test).astype(int)

    # --- Global metrics
    try: auc = roc_auc_score(y_true, y_prob)
    except: auc = float('nan')
    try: ap  = average_precision_score(y_true, y_prob)
    except: ap  = float('nan')

    # --- Thresholded metrics
    m05 = _metrics_at_threshold(y_true, y_prob, 0.5)

    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12) if thr.size else np.array([])
    best_idx = int(np.nanargmax(f1s)) if f1s.size else 0
    best_thr = float(thr[best_idx]) if thr.size else 0.5
    mbest = _metrics_at_threshold(y_true, y_prob, best_thr)

    # --- Plots: Confusion Matrices
    _save_cm(m05, f'Confusion Matrix @0.5\nAcc {m05["acc"]:.3f} F1 {m05["f1"]:.3f}',
             os.path.join(outdir, "cm_thr0.5.png"))
    _save_cm(mbest, f'Confusion Matrix @bestF1={mbest["thr"]:.3f}\nAcc {mbest["acc"]:.3f} F1 {mbest["f1"]:.3f}',
             os.path.join(outdir, "cm_bestf1.png"))

    # --- Plots: ROC & PR
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig = plt.figure(figsize=(5, 4), dpi=160); ax = plt.gca()
    ax.plot(fpr, tpr, label=f'AUC={auc:.3f}')
    ax.plot([0, 1], [0, 1], '--', linewidth=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC'); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "roc.png"), bbox_inches='tight'); plt.close(fig)

    fig = plt.figure(figsize=(5, 4), dpi=160); ax = plt.gca()
    ax.plot(rec, prec, label=f'AP={ap:.3f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('PR curve'); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "pr.png"), bbox_inches='tight'); plt.close(fig)

    # --- Save summary
    summary = dict(auc=float(auc), ap=float(ap), thr0_5=m05, best_f1=mbest)
    with open(os.path.join(outdir, "metrics.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # --- Console summary
    print(f"[{model_name}] AUC={auc:.4f} AP={ap:.4f}")
    print(f"  @0.5                : acc={m05['acc']:.3f} prec={m05['prec']:.3f} rec={m05['rec']:.3f} "
          f"f1={m05['f1']:.3f} spec={m05['spec']:.3f} mcc={m05['mcc']:.3f}")
    print(f"  @bestF1 (thr={mbest['thr']:.3f}): acc={mbest['acc']:.3f} prec={mbest['prec']:.3f} rec={mbest['rec']:.3f} "
          f"f1={mbest['f1']:.3f} spec={mbest['spec']:.3f} mcc={mbest['mcc']:.3f}")
    print(f"Artifacts written to: {outdir}")

    return summary


In [0]:
print_saved_report("xgb_baseline")
# or: print_saved_report("xgb_baseline")


In [0]:
import os, json
from IPython.display import display
from PIL import Image

model_name = "xgb_baseline"   # <- change to your run name
outdir = f"/dbfs/reports/{model_name}"

# 1) Load & print metrics
with open(os.path.join(outdir, "metrics.json"), "r") as f:
    M = json.load(f)

print(f"[{model_name}]")
print(f"AUC: {M['auc']:.4f} | AP: {M['ap']:.4f}")
m05 = M["thr0_5"]; mb = M["best_f1"]
print(" @0.5      -> "
      f"acc={m05['acc']:.3f} prec={m05['prec']:.3f} rec={m05['rec']:.3f} "
      f"f1={m05['f1']:.3f} spec={m05['spec']:.3f} mcc={m05['mcc']:.3f}")
print(f" @bestF1({mb['thr']:.3f}) -> "
      f"acc={mb['acc']:.3f} prec={mb['prec']:.3f} rec={mb['rec']:.3f} "
      f"f1={mb['f1']:.3f} spec={mb['spec']:.3f} mcc={mb['mcc']:.3f}")

# 2) Display figures
for png in ["cm_thr0.5.png", "cm_bestf1.png", "roc.png", "pr.png"]:
    path = os.path.join(outdir, png)
    print(png)
    display(Image.open(path))


In [0]:
import os, json, builtins
from datetime import datetime

def print_saved_report(model_name, base_dir="/dbfs/reports"):
    path = f"{base_dir}/{model_name}/metrics.json"
    with open(path, "r") as f:
        M = json.load(f)

    ts = datetime.fromtimestamp(os.path.getmtime(path)).strftime("%Y%m%d_%H%M%S")

    # N and positive rate from the 0.5 confusion counts
    t = M.get("thr0_5", {})
    tp, fp, tn, fn = (t.get("tp", 0), t.get("fp", 0), t.get("tn", 0), t.get("fn", 0))
    N = tp + tn + fp + fn
    pos_rate = (tp + fn) / (builtins.max(N, 1))

    def fmt(x, nd=4):
        try: return f"{x:.{nd}f}"
        except: return "nan"

    print(f"=== {model_name} | {ts} ===")
    print(f"N={N}, Pos rate={fmt(pos_rate)}")
    print(f"AUC={fmt(M.get('auc', float('nan')))}, AP={fmt(M.get('ap', float('nan')))}")

    if "thr0_5" in M:
        m05 = M["thr0_5"]
        print(f"[thr=0.50]  ACC={fmt(m05.get('acc', float('nan')))}  "
              f"PREC={fmt(m05.get('prec', float('nan')))}  "
              f"REC={fmt(m05.get('rec', float('nan')))}  "
              f"F1={fmt(m05.get('f1', float('nan')))}")

    if "best_f1" in M:
        mb = M["best_f1"]
        thr = mb.get("thr", float("nan"))
        print(f"[thr={thr:.3f}] ACC={fmt(mb.get('acc', float('nan')))}  "
              f"PREC={fmt(mb.get('prec', float('nan')))}  "
              f"REC={fmt(mb.get('rec', float('nan')))}  "
              f"F1={fmt(mb.get('f1', float('nan')))}")
